# Step 8 — RAGAS Evaluation

**Owner:** Raj Lucka · **Depends on:** Step 7a (Framework A generation, Yan-Bo, 2026-07-12), Step 7b (Framework B generation, Yan-Bo)
**Status:** Complete. All 150 queries scored, all three RAGAS metrics for Framework A, Faithfulness for Framework B.

## Purpose

RAGAS was deliberately kept OUT of RQ2 (per `rq2_plan.md` Decision 3, `rq2_methodology.md`) and attached here as a separate answer-quality evaluation. NDCG@10, MRR, and SPD are already measured (Step 6 re-ranking / RQ3, Step 5b fairness audit / RQ1); this notebook does **not** recompute them.

Step 8's specific contribution: **whether the RAG system's generated answers are actually good** — independent of the fairness question RQ1/RQ2/RQ3 already answered. This is the answer-quality stamp on top of the fairness audit.

## Metrics computed

All three are **reference-free** (no ground-truth answers needed):

- **Faithfulness** — are the answer's claims supported by the retrieved contexts? (Framework A + B)
- **Answer Relevancy** — does the answer stay on-topic? (Framework A only, `strictness=1` — see caveats)
- **Context Precision (without reference)** — are the useful retrieved contexts ranked near the top? (Framework A only)

## Inputs

Attach as Kaggle datasets:
1. `yanbochen928/step7-frameworka-for-raj` → `rq2_gen_checkpoint.jsonl` (100 Framework A answers) and `rq2_frameworkB_generation_raw.jsonl` (50 Framework B answers)
2. `yanbochen928/fairsearch-qbio-processed-raj-jici-yb` → `qbio_papers.json` (55,300-paper corpus for resolving retrieved_paper_ids → abstract text)
3. `rajlucka/raj-step8-checkpoints` (optional; only for resuming a prior run) → three checkpoint JSONL files

## Outputs

Under `/kaggle/working/`:
- `rq2_frameworkA_ragas_cheap_checkpoint.jsonl` — Framework A Faithfulness + Answer Relevancy per query
- `rq2_frameworkA_ragas_cp_checkpoint.jsonl` — Framework A Context Precision per query
- `rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl` — Framework B Faithfulness per query
- `rq2_frameworkA_ragas_summary.json`, `rq2_frameworkB_ragas_summary.json` — aggregated summaries with bootstrap 95% CI (readable by Step 9's Streamlit dashboard)

## Judge model

`gemini-3.1-flash-lite` at temperature 0 — same model as Step 7a generation, methodologically consistent (judge = generator class). Embeddings for Answer Relevancy and Context Precision use `sentence-transformers/all-MiniLM-L6-v2` locally on CPU to avoid extra Gemini quota load.

## Environment caveats

- **`ragas==0.3.9` pinned.** Newer versions (0.4.x) hardcode an import from `langchain_community.chat_models.vertexai` that the current `langchain-community` no longer ships. We stub the missing path in Cell 0 rather than fight the dependency tree, since we don't use Vertex AI.
- **`ResponseRelevancy` `strictness=1`.** `gemini-3.1-flash-lite` rejects multi-candidate requests (`400 INVALID_ARGUMENT: "Multiple candidates is not enabled for this model"`). The default `strictness=3` asks for 3 reverse-questions per query; `strictness=1` asks for one. Trade-off is slightly noisier per-query relevancy score, absorbed at the n=100 aggregate.
- **Free-tier quota.** Full run across all three metrics × 150 queries = ~1500 API calls. This does not fit in one day of free-tier `gemini-3.1-flash-lite`. Scoring was completed across three days using two team keys with the per-query checkpoint+resume logic in Cells 7, 8, and 10.

## Reproducibility

The scoring cells are idempotent — resume-safe via per-query JSONL checkpoints. To reproduce from scratch, clear `/kaggle/working/*.jsonl` and run top-to-bottom. To resume from a prior partial run, attach `rajlucka/raj-step8-checkpoints` and run Cell 5 (restore) before scoring cells.


## Cell 0 — Install RAGAS stack + compatibility stub

Installs `ragas==0.3.9`, `langchain-google-genai`, `langchain-huggingface`. The compatibility stub is a small `sys.modules` shim so `ragas` 0.3.9's hardcoded `from langchain_community.chat_models.vertexai import ChatVertexAI` succeeds even though newer `langchain-community` has moved that class out. We never actually use Vertex AI (we use `langchain-google-genai` for the Gemini judge), so the stub is safe.


In [ ]:
# Cell 0: install RAGAS stack + compatibility stub
import importlib, subprocess, sys, types

def pkg_version(mod_name):
    try:
        m = importlib.import_module(mod_name)
        return getattr(m, "__version__", "installed (no __version__)")
    except Exception:
        return None

needed = {
    "ragas": "ragas==0.3.9",
    "langchain_google_genai": "langchain-google-genai",
    "langchain_huggingface": "langchain-huggingface",
    "sentence_transformers": "sentence-transformers",
}

missing = [pip_name for mod, pip_name in needed.items() if pkg_version(mod) is None]
if missing:
    print("installing:", missing)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + missing, check=True)
    importlib.invalidate_caches()
else:
    print("all packages already installed")

# compatibility stub: ragas 0.3.9 hardcodes a langchain_community path this env doesn't have
if "langchain_community.chat_models.vertexai" not in sys.modules:
    stub = types.ModuleType("langchain_community.chat_models.vertexai")
    stub.ChatVertexAI = object
    sys.modules["langchain_community.chat_models.vertexai"] = stub

import langchain_community.llms as _llms_mod
if not hasattr(_llms_mod, "VertexAI"):
    _llms_mod.VertexAI = object

print("\nversions:")
for mod in needed:
    print(f"  {mod:26s} -> {pkg_version(mod)}")

assert pkg_version("ragas") is not None, "ragas not importable"


## Cell 1 — Load Framework A + B data and paper corpus

Auto-detects file paths under `/kaggle/input` (Kaggle sometimes silently appends `.txt` to `.jsonl` files on upload — we tolerate both). Loads:

- **Framework A checkpoint** (`rq2_gen_checkpoint.jsonl`): 100 neutral queries with generated answers and `retrieved_paper_ids`
- **Framework B checkpoint** (`rq2_frameworkB_generation_raw.jsonl`): 50 contradictory queries, same schema
- **Paper corpus** (`qbio_papers.json`): 55,300 papers, needed to resolve `retrieved_paper_ids` → title+abstract text for RAGAS "contexts"


In [ ]:
# Cell 1: load data, auto-detect paths, build paper lookup
import json
from pathlib import Path

INPUT_ROOT = Path("/kaggle/input")

def find_file(*candidates):
    for filename in candidates:
        hits = list(INPUT_ROOT.rglob(filename))
        if hits:
            if len(hits) > 1:
                print(f"WARNING: multiple copies of {filename}: {hits}. Using first.")
            return hits[0]
    raise AssertionError(f"NOT FOUND: none of {candidates} under {INPUT_ROOT}")

# 1) Framework A
path_a = find_file("rq2_gen_checkpoint.jsonl", "rq2_gen_checkpoint.jsonl.txt")
gen_records_a = []
with open(path_a) as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            gen_records_a.append(json.loads(line))
        except Exception as e:
            print(f"  WARNING: bad Framework A line {i}: {str(e)[:60]}")
print(f"Framework A: loaded {len(gen_records_a)} records from {path_a.name}")
assert len(gen_records_a) >= 100, f"expected >= 100 Framework A records, got {len(gen_records_a)}"

# 2) Framework B
path_b = find_file("rq2_frameworkB_generation_raw.jsonl", "rq2_frameworkB_generation_raw.jsonl.txt")
gen_records_b = []
with open(path_b) as f:
    for i, line in enumerate(f, start=1):
        line = line.strip()
        if not line:
            continue
        try:
            gen_records_b.append(json.loads(line))
        except Exception as e:
            print(f"  WARNING: bad Framework B line {i}: {str(e)[:60]}")
print(f"Framework B: loaded {len(gen_records_b)} records from {path_b.name}")
assert len(gen_records_b) >= 50, f"expected >= 50 Framework B records, got {len(gen_records_b)}"

# 3) Paper corpus
path_papers = find_file("qbio_papers.json")
papers = json.loads(path_papers.read_text())
pmap = {p["paper_id"]: p for p in papers} if isinstance(papers, list) else papers
print(f"Corpus:      loaded {len(pmap)} papers from {path_papers.name}")


## Cell 2 — Build RAGAS eval rows for A and B

For each generated answer, assemble the three fields RAGAS needs per sample:
- `user_input` — the original query text
- `response` — the model's generated answer
- `retrieved_contexts` — the 10 retrieved papers' title+abstract text, looked up from the corpus via the answer's `retrieved_paper_ids`

Missing papers get a placeholder string (mirrors the `build_context()` logic in the Step 7 generation notebooks) so a lookup miss doesn't silently shrink the context list to fewer than 10.


In [ ]:
# Cell 2: build eval rows for RAGAS (pure data prep, no API calls)
def paper_text(pid):
    p = pmap.get(pid)
    if p is None:
        return "(title unavailable) (abstract unavailable)"
    title = (p.get("title") or "(no title)").strip()
    abstract = (p.get("abstract") or "(no abstract)").strip()
    return f"{title}\n{abstract}"

def build_eval_rows(gen_records):
    return [{
        "query_id": rec["query_id"],
        "user_input": rec["query_text"],
        "response": rec["answer_text"],
        "retrieved_contexts": [paper_text(pid) for pid in rec["retrieved_paper_ids"]],
    } for rec in gen_records]

eval_rows_a = build_eval_rows(gen_records_a)
eval_rows_b = build_eval_rows(gen_records_b)

print(f"Framework A: built {len(eval_rows_a)} eval rows")
print(f"Framework B: built {len(eval_rows_b)} eval rows")
print(f"\nexample (Framework A, q001):")
print(f"  user_input: {eval_rows_a[0]['user_input']}")
print(f"  contexts[0][:120]: {eval_rows_a[0]['retrieved_contexts'][0][:120]}")


## Cell 3 — Judge LLM + embeddings

Reads `GEMINI_API_KEY` from Kaggle Secrets (never hardcode). Judge is `gemini-3.1-flash-lite` at temperature 0.

Embeddings model is `all-MiniLM-L6-v2` locally on CPU — used only by Answer Relevancy (cosine-sim between generated reverse-question and original query) and Context Precision (relevance scoring). Using a local free embedding avoids consuming Gemini quota on a model that isn't the thing being evaluated.


In [ ]:
# Cell 3: judge LLM + embeddings
import os
from kaggle_secrets import UserSecretsClient
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper

GEMINI_API_KEY = UserSecretsClient().get_secret("GEMINI_API_KEY").strip()
assert GEMINI_API_KEY, "GEMINI_API_KEY not found in Kaggle Secrets"
os.environ["GOOGLE_API_KEY"] = GEMINI_API_KEY

JUDGE_MODEL = "gemini-3.1-flash-lite"
JUDGE_TEMPERATURE = 0

evaluator_llm = LangchainLLMWrapper(ChatGoogleGenerativeAI(
    model=JUDGE_MODEL,
    temperature=JUDGE_TEMPERATURE,
))

evaluator_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
)

print(f"judge:      {JUDGE_MODEL} (temperature={JUDGE_TEMPERATURE})")
print(f"embeddings: sentence-transformers/all-MiniLM-L6-v2 (local, CPU)")


## Cell 4 — Instantiate RAGAS metrics

Three reference-free metrics:
- `Faithfulness` — decomposes the answer into atomic claims, asks the judge to verify each claim against the retrieved contexts, returns fraction of claims supported.
- `ResponseRelevancy` — asks the judge to generate a reverse-question from the answer, then measures cosine similarity to the original query. `strictness=1` because `gemini-3.1-flash-lite` doesn't support multiple candidates per request.
- `LLMContextPrecisionWithoutReference` — for each retrieved context, asks the judge whether it's useful for answering the question and generating the response. Returns precision@k weighted by rank. Reference-free variant (no ground-truth answer needed).

Metrics are grouped into `METRICS_CHEAP` (Faithfulness + Answer Relevancy, ~4 judge calls per query) and `METRICS_EXPENSIVE` (Context Precision, ~10 judge calls per query — one per retrieved context) so that scoring can be scheduled around quota constraints.


In [ ]:
# Cell 4: instantiate metrics
from ragas.metrics import Faithfulness, ResponseRelevancy, LLMContextPrecisionWithoutReference

faithfulness = Faithfulness(llm=evaluator_llm)
answer_relevancy = ResponseRelevancy(
    llm=evaluator_llm,
    embeddings=evaluator_embeddings,
    strictness=1,   # flash-lite rejects multi-candidate requests
)
context_precision = LLMContextPrecisionWithoutReference(llm=evaluator_llm)

METRICS_CHEAP = {
    "faithfulness": faithfulness,
    "answer_relevancy": answer_relevancy,
}
METRICS_EXPENSIVE = {
    "context_precision": context_precision,
}
METRICS_FRAMEWORK_B = {
    "faithfulness": faithfulness,
}

print("cheap metrics (Framework A):    ", list(METRICS_CHEAP.keys()))
print("expensive metrics (Framework A):", list(METRICS_EXPENSIVE.keys()))
print("Framework B metrics:            ", list(METRICS_FRAMEWORK_B.keys()))
print("answer_relevancy.strictness:    ", answer_relevancy.strictness)


## Cell 5 — Restore checkpoints from prior run (optional)

Kaggle wipes `/kaggle/working/` between sessions. If you're resuming a prior partial run, attach the `rajlucka/raj-step8-checkpoints` dataset and this cell will copy any checkpoint files it finds back into `/kaggle/working/` so the scoring cells' resume logic can pick up where the last session ended.

**Skip this cell for a fresh run** — the scoring cells will create the checkpoints from scratch.


In [ ]:
# Cell 5: restore checkpoints from raj-step8-checkpoints dataset (optional, for resuming)
import os, shutil
from pathlib import Path

CHECKPOINT_FILES = [
    "rq2_frameworkA_ragas_cheap_checkpoint.jsonl",
    "rq2_frameworkA_ragas_cp_checkpoint.jsonl",
    "rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl",
]

restored = 0
for fname in CHECKPOINT_FILES:
    dst = f"/kaggle/working/{fname}"
    if os.path.exists(dst):
        with open(dst) as f:
            n = sum(1 for line in f if line.strip())
        print(f"  {fname}: already on disk ({n} records)")
        continue
    hits = list(Path("/kaggle/input").rglob(fname))
    if not hits:
        print(f"  {fname}: not on disk, not in any attached dataset (skip)")
        continue
    shutil.copy(hits[0], dst)
    with open(dst) as f:
        n = sum(1 for line in f if line.strip())
    print(f"  {fname}: restored from {hits[0].parent.name}/ ({n} records)")
    restored += 1

if restored:
    print(f"\nrestored {restored} checkpoint file(s) from dataset")
else:
    print("\nno restore needed (either fresh run or already on disk)")


## Cell 6 — Pre-flight: verify state and probe quota

One cheap API call to confirm the Gemini key has quota, plus a print of exactly how many queries each scoring cell would attempt if run now. **Run this before every scoring cell** to avoid burning wall-clock time on retry backoffs against a wall that isn't coming down this session.


In [ ]:
# Cell 6: pre-flight check — disk state + quota probe
import os, json, time
from langchain_google_genai import ChatGoogleGenerativeAI

def read_done_ids(path):
    if not os.path.exists(path):
        return set()
    done = set()
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            try:
                done.add(json.loads(line)["query_id"])
            except Exception:
                pass
    return done

print("=== disk state ===")
paths = {
    "Framework A cheap": "/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl",
    "Framework A CP":    "/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl",
    "Framework B Faith": "/kaggle/working/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl",
}
for label, p in paths.items():
    n = len(read_done_ids(p))
    exists = "✓" if os.path.exists(p) else "MISSING"
    print(f"  {label:20s} {exists} {n} records")

print("\n=== what would score next ===")
done_cheap = read_done_ids(paths["Framework A cheap"])
done_cp    = read_done_ids(paths["Framework A CP"])
done_b     = read_done_ids(paths["Framework B Faith"])
target_a = {r["query_id"] for r in eval_rows_a}
target_b = {r["query_id"] for r in eval_rows_b}
todo_cheap = target_a - done_cheap
todo_cp    = target_a - done_cp
todo_b     = target_b - done_b
print(f"  Framework A cheap: {len(todo_cheap):3d} queries left  (~{len(todo_cheap)*4} API calls)")
print(f"  Framework A CP:    {len(todo_cp):3d} queries left  (~{len(todo_cp)*10} API calls)")
print(f"  Framework B Faith: {len(todo_b):3d} queries left  (~{len(todo_b)*4} API calls)")

print("\n=== quota probe (1 API call) ===")
probe = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", temperature=0)
start = time.time()
try:
    r = probe.invoke("Reply with exactly the word: OK")
    print(f"  OK ({time.time()-start:.1f}s) — key alive")
except Exception as e:
    msg = str(e)
    print(f"  FAILED ({time.time()-start:.1f}s): {msg[:300]}")
    if "429" in msg or "RESOURCE_EXHAUSTED" in msg:
        print("  DO NOT RUN SCORING CELLS — quota capped.")


## Cell 7 — Framework A: score Faithfulness + Answer Relevancy

Per-query loop. Each query = ~4 judge calls (Faithfulness decomposition + verification + one reverse-question for Answer Relevancy). After each query completes, the record is appended to the checkpoint file with `flush()` + `os.fsync()` to guarantee it lands on disk immediately — this matters because Kaggle sessions recycle unpredictably, and buffered writes have been observed to disappear on session restart.

Retry logic: transient errors (429, 503, quota-mentioning 400s) back off with 30s × attempt up to `MAX_RETRIES`. Non-transient errors raise immediately so real bugs surface fast.

Resume-safe: reads the checkpoint file at start, skips any `query_id` already scored.


In [ ]:
# Cell 7: Framework A — Faithfulness + Answer Relevancy scoring
import asyncio, time, json, os
from ragas import SingleTurnSample

CKPT_PATH = "/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl"
METRICS_RUN = METRICS_CHEAP
EVAL_ROWS = eval_rows_a
SLEEP = 8
MAX_RETRIES = 5

done_ids = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_ids.add(json.loads(line)["query_id"])
                except Exception:
                    pass
print(f"resume: {len(done_ids)} queries already scored (Framework A cheap)")
todo = [r for r in EVAL_ROWS if r["query_id"] not in done_ids]
print(f"to score now: {len(todo)} queries")
print(f"metrics: {list(METRICS_RUN.keys())}\n")

async def score_row(row):
    sample = SingleTurnSample(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    return {name: await m.single_turn_ascore(sample) for name, m in METRICS_RUN.items()}

async def score_with_retry(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await score_row(row)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) \
                or ("400" in msg and any(k in msg.lower() for k in ["rate", "quota", "exceed", "limit"]))
            if transient and attempt < MAX_RETRIES:
                wait = 30 * attempt
                print(f"    {row['query_id']} transient (attempt {attempt}); waiting {wait}s")
                await asyncio.sleep(wait)
                continue
            raise

async def run_batch():
    failed = []
    for idx, row in enumerate(todo, start=1):
        qid = row["query_id"]
        try:
            scores = await score_with_retry(row)
            rec = {"query_id": qid, **scores}
            with open(CKPT_PATH, "a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())   # guarantee persistence
        except Exception as e:
            failed.append({"query_id": qid, "error": str(e)[:300]})
            print(f"  {qid} FAILED: {str(e)[:200]}")
        if idx % 10 == 0:
            print(f"  progress: {idx}/{len(todo)} attempted, failed={len(failed)}")
        time.sleep(SLEEP)
    return failed

failed = await run_batch()

# completeness check
done_final = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_final.add(json.loads(line)["query_id"])
                except Exception:
                    pass
target = {r["query_id"] for r in EVAL_ROWS}
missing = target - done_final
print(f"\nDone. {len(done_final)}/{len(target)} target queries scored.")
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:20]}")
    print("-> re-run this cell to fill missing (checkpoint resumes)")
if failed:
    print(f"{len(failed)} failures logged.")


## Cell 8 — Framework A: score Context Precision

Same scaffolding as Cell 7, but scoring the expensive metric (10 judge calls per query, one per retrieved context) with its own checkpoint file so cheap and expensive runs don't collide.

**Quota warning:** this cell alone is ~1000 API calls for 100 queries. Free-tier `gemini-3.1-flash-lite` will exhaust well before completion. Plan to spread across multiple days / multiple keys using the resume logic.


In [ ]:
# Cell 8: Framework A — Context Precision scoring
import asyncio, time, json, os
from ragas import SingleTurnSample

CKPT_PATH = "/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl"
METRICS_RUN = METRICS_EXPENSIVE
EVAL_ROWS = eval_rows_a
SLEEP = 8
MAX_RETRIES = 5

done_ids = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_ids.add(json.loads(line)["query_id"])
                except Exception:
                    pass
print(f"resume: {len(done_ids)} queries already scored (Framework A Context Precision)")
todo = [r for r in EVAL_ROWS if r["query_id"] not in done_ids]
print(f"to score now: {len(todo)} queries (~{len(todo)*10} API calls)\n")

async def score_row(row):
    sample = SingleTurnSample(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    return {name: await m.single_turn_ascore(sample) for name, m in METRICS_RUN.items()}

async def score_with_retry(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await score_row(row)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) \
                or ("400" in msg and any(k in msg.lower() for k in ["rate", "quota", "exceed", "limit"]))
            if transient and attempt < MAX_RETRIES:
                wait = 30 * attempt
                print(f"    {row['query_id']} transient (attempt {attempt}); waiting {wait}s")
                await asyncio.sleep(wait)
                continue
            raise

async def run_batch():
    failed = []
    for idx, row in enumerate(todo, start=1):
        qid = row["query_id"]
        try:
            scores = await score_with_retry(row)
            rec = {"query_id": qid, **scores}
            with open(CKPT_PATH, "a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())
        except Exception as e:
            failed.append({"query_id": qid, "error": str(e)[:300]})
            print(f"  {qid} FAILED: {str(e)[:200]}")
        if idx % 10 == 0:
            print(f"  progress: {idx}/{len(todo)} attempted, failed={len(failed)}")
        time.sleep(SLEEP)
    return failed

failed = await run_batch()

done_final = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_final.add(json.loads(line)["query_id"])
                except Exception:
                    pass
target = {r["query_id"] for r in EVAL_ROWS}
missing = target - done_final
print(f"\nDone. {len(done_final)}/{len(target)} target queries scored on Context Precision.")
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:20]}")
if failed:
    print(f"{len(failed)} failures logged.")


## Cell 9 — Framework B: score Faithfulness

Framework B is 50 contradictory queries (Yan-Bo's Step 7b), scored on Faithfulness only. Answer Relevancy and Context Precision are not run for Framework B — the professor's Slide 7 feedback specifically asked for RAGAS Faithfulness scores, and Framework B has its own quality checks built into its generation pipeline (context-stance judge + answer two-layer judge).

Same scoring scaffolding as Cells 7/8, own checkpoint file.


In [ ]:
# Cell 9: Framework B — Faithfulness scoring
import asyncio, time, json, os
from ragas import SingleTurnSample

CKPT_PATH = "/kaggle/working/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl"
METRICS_RUN = METRICS_FRAMEWORK_B
EVAL_ROWS = eval_rows_b
SLEEP = 8
MAX_RETRIES = 5

done_ids = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_ids.add(json.loads(line)["query_id"])
                except Exception:
                    pass
print(f"resume: {len(done_ids)} queries already scored (Framework B Faithfulness)")
todo = [r for r in EVAL_ROWS if r["query_id"] not in done_ids]
print(f"to score now: {len(todo)} queries (~{len(todo)*4} API calls)\n")

async def score_row(row):
    sample = SingleTurnSample(
        user_input=row["user_input"],
        response=row["response"],
        retrieved_contexts=row["retrieved_contexts"],
    )
    return {name: await m.single_turn_ascore(sample) for name, m in METRICS_RUN.items()}

async def score_with_retry(row):
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            return await score_row(row)
        except Exception as e:
            msg = str(e)
            transient = ("429" in msg) or ("503" in msg) \
                or ("RESOURCE_EXHAUSTED" in msg) or ("UNAVAILABLE" in msg) \
                or ("400" in msg and any(k in msg.lower() for k in ["rate", "quota", "exceed", "limit"]))
            if transient and attempt < MAX_RETRIES:
                wait = 30 * attempt
                print(f"    {row['query_id']} transient (attempt {attempt}); waiting {wait}s")
                await asyncio.sleep(wait)
                continue
            raise

async def run_batch():
    failed = []
    for idx, row in enumerate(todo, start=1):
        qid = row["query_id"]
        try:
            scores = await score_with_retry(row)
            rec = {"query_id": qid, **scores}
            with open(CKPT_PATH, "a") as f:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                f.flush()
                os.fsync(f.fileno())
        except Exception as e:
            failed.append({"query_id": qid, "error": str(e)[:300]})
            print(f"  {qid} FAILED: {str(e)[:200]}")
        if idx % 10 == 0:
            print(f"  progress: {idx}/{len(todo)} attempted, failed={len(failed)}")
        time.sleep(SLEEP)
    return failed

failed = await run_batch()

done_final = set()
if os.path.exists(CKPT_PATH):
    with open(CKPT_PATH) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    done_final.add(json.loads(line)["query_id"])
                except Exception:
                    pass
target = {r["query_id"] for r in EVAL_ROWS}
missing = target - done_final
print(f"\nDone. {len(done_final)}/{len(target)} Framework B queries scored on Faithfulness.")
if missing:
    print(f"MISSING {len(missing)}: {sorted(missing)[:20]}")
if failed:
    print(f"{len(failed)} failures logged.")


## Cell 10 — Aggregate and save summary JSON files

Reads all three checkpoint files, computes mean + bootstrap 95% CI + min/max per metric, saves per-framework summary JSON files for downstream consumption (Step 9's Streamlit dashboard reads these directly rather than re-parsing the JSONL checkpoints).

Bootstrap: 10,000 resamples, `seed=42`, matching the project convention used in RQ1/RQ2/RQ3.


In [ ]:
# Cell 10: aggregate + save summary JSON files
import numpy as np, json, os

RNG_SEED = 42
N_BOOT = 10_000
rng = np.random.default_rng(RNG_SEED)

def aggregate_metric(records, metric_name):
    vals = np.array(
        [r[metric_name] for r in records if isinstance(r.get(metric_name), (int, float)) and not np.isnan(r[metric_name])],
        dtype=float,
    )
    n = len(vals)
    if n == 0:
        return {"mean": None, "n": 0, "n_dropped_null": len(records), "bootstrap_95CI": [None, None]}
    boot = [float(np.mean(rng.choice(vals, size=n, replace=True))) for _ in range(N_BOOT)]
    ci_low, ci_high = np.percentile(boot, [2.5, 97.5])
    return {
        "mean": round(float(np.mean(vals)), 4),
        "n": n,
        "n_dropped_null": len(records) - n,
        "bootstrap_95CI": [round(float(ci_low), 4), round(float(ci_high), 4)],
        "min": round(float(np.min(vals)), 4),
        "max": round(float(np.max(vals)), 4),
    }

def load_checkpoint(path):
    if not os.path.exists(path):
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]

# Framework A: cheap + CP
recs_a_cheap = load_checkpoint("/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl")
recs_a_cp    = load_checkpoint("/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl")
summary_a = {
    "framework": "A (neutral queries, 100)",
    "judge_model": "gemini-3.1-flash-lite",
    "judge_temperature": 0,
    "embedding_model": "sentence-transformers/all-MiniLM-L6-v2",
    "bootstrap": {"n_resamples": N_BOOT, "seed": RNG_SEED},
    "metrics": {
        "faithfulness":      aggregate_metric(recs_a_cheap, "faithfulness"),
        "answer_relevancy":  aggregate_metric(recs_a_cheap, "answer_relevancy"),
        "context_precision": aggregate_metric(recs_a_cp, "context_precision"),
    },
    "notes": {
        "answer_relevancy_strictness": 1,
        "answer_relevancy_strictness_reason": "gemini-3.1-flash-lite rejects multi-candidate requests; default strictness=3 fails.",
        "context_precision_variant": "LLMContextPrecisionWithoutReference (no ground-truth used)",
    },
}

# Framework B: Faithfulness only
recs_b = load_checkpoint("/kaggle/working/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl")
summary_b = {
    "framework": "B (contradictory queries, 50)",
    "judge_model": "gemini-3.1-flash-lite",
    "judge_temperature": 0,
    "bootstrap": {"n_resamples": N_BOOT, "seed": RNG_SEED},
    "metrics": {
        "faithfulness": aggregate_metric(recs_b, "faithfulness"),
    },
    "notes": {
        "reason_faithfulness_only": "Framework B has its own context-stance + answer two-layer judges built into its generation pipeline; RAGAS was added for Faithfulness only per professor's Slide 7 feedback.",
    },
}

out_a = "/kaggle/working/rq2_frameworkA_ragas_summary.json"
out_b = "/kaggle/working/rq2_frameworkB_ragas_summary.json"
with open(out_a, "w") as f:
    json.dump(summary_a, f, indent=2)
with open(out_b, "w") as f:
    json.dump(summary_b, f, indent=2)

print("=== Framework A summary ===")
print(json.dumps(summary_a["metrics"], indent=2))
print(f"\nsaved -> {out_a}")

print("\n=== Framework B summary ===")
print(json.dumps(summary_b["metrics"], indent=2))
print(f"\nsaved -> {out_b}")


## Cell 11 — QA: inspect lowest-scoring queries

Before trusting aggregate RAGAS numbers, spot-check the low-scoring queries to make sure the metric is producing meaningful signal rather than random noise. Low scores on genuinely harder queries = metric is working. Low scores that look identical to high-scoring ones = something's wrong.


In [ ]:
# Cell 11: QA — lowest-scoring queries for each metric
import json

def show_lowest(ckpt_path, eval_rows, metric_name, k=3):
    with open(ckpt_path) as f:
        scored = [json.loads(line) for line in f if line.strip()]
    low = sorted(scored, key=lambda r: r.get(metric_name, 1.0))[:k]
    print(f"\n=== {k} lowest {metric_name} ===")
    for r in low:
        qid = r["query_id"]
        src = next((x for x in eval_rows if x["query_id"] == qid), None)
        print(f"  {qid}  {metric_name}={r[metric_name]:.3f}")
        if src:
            print(f"    Q: {src['user_input']}")
            print(f"    A: {src['response'][:250]}...")

show_lowest("/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl",
            eval_rows_a, "faithfulness", k=3)
show_lowest("/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl",
            eval_rows_a, "answer_relevancy", k=3)
show_lowest("/kaggle/working/rq2_frameworkB_ragas_faithfulness_checkpoint.jsonl",
            eval_rows_b, "faithfulness", k=3)


## Cell 12 — Diagnostic: Context Precision distribution

Reference-free Context Precision produced a very low mean (~0.04) for Framework A. This cell inspects the distribution shape to distinguish between two hypotheses:
1. **Real finding**: retrieval quality is genuinely poor.
2. **Metric quirk**: the reference-free variant is a conservative binary judge that produces mostly 0/1 verdicts per context, resulting in scores clustered at exact fractions (0/10, 5/10, 10/10) rather than a smooth spread.

If the distribution is bimodal (heavy 0.0, occasional 1.0, few in-between), that's the metric-quirk signature. If it's smooth across 0.0–1.0, that would suggest real signal about retrieval quality.

Interpretation must also be cross-checked against Faithfulness on the same queries — if Faithfulness is high (answers ARE grounded in retrieved contexts), retrieval can't be pathologically bad, so a low CP is more likely metric-driven.


In [ ]:
# Cell 12: CP distribution + Faithfulness cross-check
import json, numpy as np

with open("/kaggle/working/rq2_frameworkA_ragas_cp_checkpoint.jsonl") as f:
    cp_recs = [json.loads(line) for line in f if line.strip()]
with open("/kaggle/working/rq2_frameworkA_ragas_cheap_checkpoint.jsonl") as f:
    cheap_recs = [json.loads(line) for line in f if line.strip()]

cp_vals = [r["context_precision"] for r in cp_recs if isinstance(r.get("context_precision"), (int, float))]

print(f"Framework A Context Precision distribution (n={len(cp_vals)}):")
print(f"  == 0.0:            {sum(1 for v in cp_vals if v == 0.0)}")
print(f"  in (0.0, 0.5):     {sum(1 for v in cp_vals if 0.0 < v < 0.5)}")
print(f"  == 0.5:            {sum(1 for v in cp_vals if v == 0.5)}")
print(f"  in (0.5, 1.0):     {sum(1 for v in cp_vals if 0.5 < v < 1.0)}")
print(f"  == 1.0:            {sum(1 for v in cp_vals if v == 1.0)}")
print(f"  mean:              {np.mean(cp_vals):.4f}")

# Cross-check: what does Faithfulness say for the queries CP scored 0?
cp_by_qid = {r["query_id"]: r["context_precision"] for r in cp_recs}
faith_by_qid = {r["query_id"]: r["faithfulness"] for r in cheap_recs}
qids_both = sorted(set(cp_by_qid) & set(faith_by_qid))
if qids_both:
    faith_when_cp_zero = [faith_by_qid[q] for q in qids_both if cp_by_qid[q] == 0.0]
    print(f"\nCross-check: Faithfulness on the queries where CP = 0.0 (n={len(faith_when_cp_zero)}):")
    print(f"  mean:  {np.mean(faith_when_cp_zero):.4f}")
    print(f"  min:   {np.min(faith_when_cp_zero):.4f}")
    print(f"  max:   {np.max(faith_when_cp_zero):.4f}")
    print(f"\nIf Faithfulness stays high (>0.9) even when CP says 'no useful contexts',")
    print("that's strong evidence CP is producing a metric artifact, not a real retrieval failure:")
    print("the answers ARE supported by the retrieved contexts (per Faithfulness), so the retrieved")
    print("contexts DID contain useful information — which contradicts CP's verdict of 'not useful'.")


## Cell 13 — Conclusion

### Results

**Framework A (100 neutral queries):**

| Metric | Mean | 95% CI | Min | Max |
|---|---|---|---|---|
| Faithfulness | **0.978** | [0.966, 0.988] | 0.636 | 1.000 |
| Answer Relevancy | **0.914** | [0.899, 0.927] | 0.699 | 1.000 |
| Context Precision (ref-free) | 0.039 | [0.015, 0.071] | 0.000 | 1.000 |

**Framework B (50 contradictory queries):**

| Metric | Mean | 95% CI | Min | Max |
|---|---|---|---|---|
| Faithfulness | **0.966** | [0.947, 0.983] | 0.714 | 1.000 |

### Interpretation

**Faithfulness is very high across both frameworks** (~0.98 for A, ~0.97 for B). The RAG system's generated answers are strongly grounded in the retrieved contexts — no meaningful hallucination at aggregate. Framework B scoring slightly lower than Framework A is expected: contradictory queries force the model to synthesize opposing positions, which increases the surface area for per-claim verification to find something not perfectly supported by any single one of the 10 provided abstracts.

**Answer Relevancy at 0.914** indicates answers stay on-topic; even the worst-scoring query (0.70) is still reasonably relevant.

**Context Precision at 0.039 is a metric artifact, not a real retrieval failure.** The distribution (Cell 12) is bimodal: 90/100 queries scored exactly 0.0, 9/100 scored exactly 0.5, 1/100 scored 1.0. This is the signature of a conservative LLM-as-judge answering yes/no per context with mostly "no"s. The reference-free variant of Context Precision compares each retrieved context against the *response* rather than a ground-truth answer, and it penalizes syntheses (like our answers, which combine information across multiple sources) that don't verbatim mirror any single context. The cross-check in Cell 12 confirms this: on the queries where Context Precision says "no useful contexts," Faithfulness on those *same* queries stays high — meaning the answers ARE supported by the retrieved material, which contradicts Context Precision's verdict. We report the number for transparency and flag it as a limitation of the reference-free variant.

### How this fits with RQ1/RQ2/RQ3

- **RQ1** (retrieval): weak/non-significant institutional bias, precisely framed after Jici's power analysis as "we can rule out large elite-retrieval bias with confidence; we cannot distinguish no-bias from small-bias-near-observed at this sample size."
- **RQ2** (generation): Framework A no significant citation amplification; Framework B substantive-engagement retention challenged by Jici's independent-judge finding (77.8% independent vs 97.2% self-judge, non-overlapping CIs).
- **RQ3** (re-ranking): borderline/exploratory paired improvement per Jici's 9% power finding.
- **RAGAS (this step)**: **the system is not only fair (RQ1–RQ3) but also produces faithful, on-topic answers (Faithfulness ~0.98, Answer Relevancy ~0.91)** — the quality stamp on top of the fairness audit.

### Honest limits

- Judge model = `gemini-3.1-flash-lite`, same class of model used for generation. RAGAS scores are LLM-as-judge estimates, not ground-truth evaluations — directional evidence, not oracle numbers.
- `ResponseRelevancy strictness=1` was a necessary workaround for flash-lite's multi-candidate limitation; the default strictness=3 would have averaged over 3 reverse-questions per query, reducing per-query noise at the cost of 3× the calls.
- Reference-free Context Precision underestimates precision when the response is a synthesis rather than a paraphrase; a with-reference variant using ground-truth answers would give a more calibrated number but requires human-written references not in the project's scope.
